In [ ]:
# %%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
# needed as this function doesn't like it when the lm_head has its size changed
from unsloth import tokenizer_utils
def do_nothing(*args, **kwargs):
    pass
tokenizer_utils.fix_untrained_tokens = do_nothing

In [ ]:
from unsloth import FastLanguageModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

In [ ]:
model_name = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(model_name, token=os.getenv("HF_TOKEN"))

In [ ]:
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    load_in_4bit=True,
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pwd

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/datasets/Symptom2Disease.csv")
df.info()

In [ ]:
labels = df.label.unique()
labels

In [ ]:
classification_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    load_in_4bit=True,
    token=os.getenv("HF_TOKEN"),
)

In [ ]:
classification_model = prepare_model_for_kbit_training(classification_model)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

In [ ]:
classification_model = get_peft_model(classification_model, lora_config)

In [ ]:
def print_trainable_params(model):
    trainable = 0
    total = 0
    for _, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    print(f"Trainable params: {trainable:,} ({100*trainable/total:.2f}% of total {total:,})")

print_trainable_params(classification_model)

In [ ]:
ds

## Stratify Split DS on labels

In [ ]:
ds = load_dataset("csv", data_files="/content/drive/MyDrive/datasets/Symptom2Disease.csv")

In [ ]:
labels = sorted(list(set(ds["train"]["label"])))
label_to_id = {l:i for i,l in enumerate(labels)}
id_to_label = {i:l for l,i in label_to_id.items()}

def encode_labels(ex):
    ex["labels"] = label_to_id[ex["label"]]
    return ex

ds = ds.map(encode_labels)

In [ ]:
ds["train"][0]

In [ ]:
# def swap_columns(example):
#     old_label = example["label"]     # string
#     old_labels = example["labels"]   # int

#     example["label"] = old_labels    # int
#     example["labels"] = old_label    # string
#     return example

# ds = ds.map(swap_columns)

In [ ]:
# df = ds["train"].to_pandas()
# df["label"].apply(type).value_counts()

In [ ]:
def tokenize_fn(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding=True,
        max_length=512,
    )

ds = ds.map(tokenize_fn, batched=True)

In [ ]:
from sklearn.model_selection import train_test_split

df = ds["train"].to_pandas()

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    shuffle=True,
    stratify=df["labels"],   #numeric labels
    random_state=42
)

from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
test_ds  = Dataset.from_pandas(test_df)

In [ ]:
train_ds[0]

In [ ]:
train_ds

In [ ]:
cols_to_remove = ["Unnamed: 0", "text", "label"]
train_ds = train_ds.remove_columns(cols_to_remove)
test_ds = test_ds.remove_columns(cols_to_remove)

In [ ]:
train_ds[0]

# Training

In [ ]:
!pip install evaluate

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels, average="weighted")["precision"],
        "recall": recall.compute(predictions=preds, references=labels, average="weighted")["recall"],
        "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

training_args = TrainingArguments(
    output_dir="llama3.2_medical_classifier",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    report_to="none",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
)

trainer = Trainer(
    model=classification_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,   # stratified test split
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
!zip -r /content/llama_sgd.zip /content/llama3.2_medical_classifier

In [ ]:
!pip install lime

In [ ]:
import gc
gc.collect()

In [ ]:
from lime.lime_text import LimeTextExplainer
import numpy as np

explainer = LimeTextExplainer(class_names=labels)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def predict_proba(texts):
    """
    LIME sends a list of raw texts.
    We must:
    1. tokenize
    2. pass through model
    3. return probabilities (numpy)
    """
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    classification_model.half()
    classification_model.to(device)

    with torch.no_grad():
        outputs = classification_model(**enc)
        probs = torch.softmax(outputs.logits, dim=1)

    return probs.cpu().numpy()


#sample text
sample = ds["train"][0]["text"]
sample_label = ds["train"][0]["labels"]

exp = explainer.explain_instance(
    sample,
    predict_proba,
    num_features=10,
    top_labels=1,
    num_samples=100
)

print("Original text:")
print(sample)
print("\nLIME Explanation:")
print(exp.as_list(label=exp.top_labels[0]))

# Visual HTML
exp.show_in_notebook(text=sample)